## Extending the example for reaching a sequence of targets

BSD 3-Clause License

Copyright (c) 2021, Nicolas Mansard
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its
   contributors may be used to endorse or promote products derived from
   this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

Now we ask you to modify this example to reach a sequence of targets $p_{*1},p_{*2},p_{*3},p_{*4}$. The optimal trajectory should use similar regularization, but with the reaching cost now varying it time: for the first quarter of nodes, the target is $p_{*1}$, then another quarter with $p_{*2}$ etc until the four targets are reached. Don't specify a particular velocity when reaching the point to let more freedom to the solver.

Below is a quick guideline to help you.

### First step: prepare the environment

Start by defining several targets (let's say 4 targets, all at x=0.4, and at y and z being either 0 or 0.4), and display then in the viewer.

In [1]:
# %load tp5/generated/panda_reaches_a_single_target_import
import crocoddyl
import example_robot_data as robex
import numpy as np
import pinocchio as pin

from supaero2025.meshcat_viewer_wrapper import MeshcatVisualizer

In [2]:
# First, let's load the Pinocchio model for the Panda arm.
robot = robex.load('panda')
# The 2 last joints are for the fingers, not important in arm motion, freeze them
robot.model,[robot.visual_model,robot.collision_model] = \
    pin.buildReducedModel(robot.model,[robot.visual_model,robot.collision_model],[8,9],robot.q0)
robot.q0 = robot.q0[:7].copy()

robot_model = robot.model
robot_model.armature = np.ones(robot.model.nv)*2 # Arbitrary value representing the true armature
robot_model.q0 = robot.q0.copy()
robot_model.x0 = np.concatenate([robot_model.q0, np.zeros(robot_model.nv)])

In [3]:
target_positions = [
    np.array([0.4, -0.1, 0.4]),
    np.array([0.4, -0.1, 0.2]),
    np.array([0.4, 0.1, 0.4]),
    np.array([0.4, 0.1, 0.2]),
]

target_placements = [
    pin.SE3(pin.utils.rpyToMatrix(-np.pi, 0, 0), p) for p in target_positions
]

viz = MeshcatVisualizer(robot)
for i, M in enumerate(target_placements):
    name = f"world/goal{i}"
    viz.addBox(name, [0.05, 0.05, 0.05], [0, 1, 0, 1])
    viz.applyConfiguration(name, M)

viz.display(robot_model.q0)

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7014/static/


In [4]:
viz.viewer.jupyter_cell()

In [9]:
# Run this cell after solving the problem
viz.play([x[:robot.model.nq] for x in ddp.xs], TIME_STEP)

### Second step: define the shooting problem

The shooting problem will be composed of 4 sequences of action models. Each sequence consists on T shooting "running" nodes and 1 terminal node. The running nodes mostly have regularization terms, while the terminal nodes have a strong cost toward the respective target.

$[ R_1,R_1,R_1 ... R_1,T_1, R_2,R_2 .... R_2, T_2, R_3 ... R_3, T_3, R_4 ... R_4 ] , T4

First create 4 running models and 4 terminal models.

In [5]:
# Initialize lists to store models for each sequence
runningModels = []
terminalModels = []
T = 100  # Number of nodes per sequence (Time per target = T * TIME_STEP)
REACH_DIMENSION = "3d"  # "6d"
TIME_STEP = 1e-2
FRAME_TIP = robot.model.getFrameId("panda_hand_tcp")
state = crocoddyl.StateMultibody(robot_model)
actuationModel = crocoddyl.ActuationModelFull(state)

Then you need to add a position cost, and state and control regularization to each running action model. Please  note that for terminal action model is only needed the position cost. Additionally, in the running models, the position cost should be low, and it should be high in the terminal models.

In [6]:
for i in range(4):
    runningCostModel = crocoddyl.CostModelSum(state)
    terminalCostModel = crocoddyl.CostModelSum(state)

    if REACH_DIMENSION == "3d":
        # Cost for 3d tracking || p(q) - pref ||**2
        goalTrackingRes = crocoddyl.ResidualModelFrameTranslation(state,FRAME_TIP,target_positions[i])
        goalTrackingWeights = crocoddyl.ActivationModelWeightedQuad(np.array([1,1,1]))
    elif REACH_DIMENSION == "6d":
        # Cost for 6d tracking  || log( M(q)^-1 Mref ) ||**2
        goalTrackingRes = crocoddyl.ResidualModelFramePlacement(state,FRAME_TIP,target_placements[i])
        goalTrackingWeights = crocoddyl.ActivationModelWeightedQuad(np.array([1,1,1, .1,.1,.1]))
    else:
        assert( REACH_DIMENSION=="3d" or REACH_DIMENSION=="6d" )
    goalTrackingCost = crocoddyl.CostModelResidual(state,goalTrackingWeights,goalTrackingRes)
    runningCostModel.addCost("gripperPose", goalTrackingCost, .1)
    terminalCostModel.addCost("gripperPose", goalTrackingCost, 4)

    # Regularization is stronger on position than velocity (to account for typical unit scale)
    xRegWeights = crocoddyl.ActivationModelWeightedQuad(np.array([1,1,1,1,1,1,1, .1,.1,.1,.1,.1,.1,.1]))
    xRegRes = crocoddyl.ResidualModelState(state,robot_model.x0)
    xRegCost = crocoddyl.CostModelResidual(state,xRegWeights,xRegRes)
    runningCostModel.addCost("xReg", xRegCost, 1e-3)

    # Terminal cost for state regularization || x - x* ||**2
    # Require more strictly a small velocity at task end (but we don't care for the position)
    xRegWeightsT=crocoddyl.ActivationModelWeightedQuad(np.array([.5,.5,.5,.5,.5,.5,.5,  5.,5.,5.,5.,5.,5.,5.]))
    xRegResT = crocoddyl.ResidualModelState(state,robot_model.x0)
    xRegCostT = crocoddyl.CostModelResidual(state,xRegWeightsT,xRegResT)
    terminalCostModel.addCost("xReg", xRegCostT, .01)

    # Cost for control regularization || u - g(q) ||**2
    uRegRes = crocoddyl.ResidualModelControlGrav(state)
    uRegCost = crocoddyl.CostModelResidual(state,uRegRes)
    runningCostModel.addCost("uReg", uRegCost, 1e-6)

    # The actuation model is here trivial: tau_q = u.
    actuationModel = crocoddyl.ActuationModelFull(state)
    # Running model composing the costs, the differential equations of motion and the integrator.
    runningModel = crocoddyl.IntegratedActionModelEuler(
        crocoddyl.DifferentialActionModelFreeFwdDynamics(state, actuationModel, runningCostModel), TIME_STEP)
    runningModel.differential.armature = robot_model.armature
    # Terminal model following the same logic, although the integration is here trivial.
    terminalModel = crocoddyl.IntegratedActionModelEuler(
        crocoddyl.DifferentialActionModelFreeFwdDynamics(state, actuationModel, terminalCostModel), 0.)
    terminalModel.differential.armature = robot_model.armature
    runningModels.append(runningModel)
    terminalModels.append(terminalModel)

Now create a shooting problem.

In [7]:
seq0 = [runningModels[0]]*T + [terminalModels[0]]
seq1 = [runningModels[1]]*T + [terminalModels[1]]
seq2 = [runningModels[2]]*T + [terminalModels[2]]
seq3 = [runningModels[3]]*T 
problem = crocoddyl.ShootingProblem(robot_model.x0,seq0+seq1+seq2+seq3,terminalModels[3])

Create a DDP solver for this problem and run it. 

In [8]:
ddp = crocoddyl.SolverDDP(problem)
ddp.solve()

True

In [10]:
# Assertion through pinocchio's forwardKinematics algorithm
xs = ddp.xs
q_traj = [x[:robot.model.nq] for x in xs]


idx1 = T
idx2 = 2*T+1
idx3 = 3*T+2
idx4 = 4*T+3

for name, idx, target in zip(
    ["wp1", "wp2", "wp3", "wp4"],
    [idx1, idx2, idx3, idx4],
    target_positions,
):
    q = q_traj[idx]
    pin.forwardKinematics(robot.model, robot.data, q)
    pin.updateFramePlacements(robot.model, robot.data)
    oMf = robot.data.oMf[FRAME_TIP]
    print(name, " reached:", oMf.translation, " target:", target)


wp1  reached: [ 0.39419531 -0.09828894  0.30745546]  target: [ 0.4 -0.1  0.4]
wp2  reached: [0.39879709 0.00212356 0.31532661]  target: [ 0.4 -0.1  0.2]
wp3  reached: [0.38995573 0.09539114 0.31390339]  target: [0.4 0.1 0.4]
wp4  reached: [0.39778752 0.09907775 0.20326137]  target: [0.4 0.1 0.2]


Well, it should not work, at least no on the first shot. The DDP solver is likely not strong enough to accept the random weights that you have selected. 

If it is working nicely from the first shot, display it in the viewer and go take a coffee. But you will likely have to tweak the gains to make it work.

**It is suggested to first optimize only sequence 1. When you are happy with it, add sequence 2 and optimize again, etc.**


### Toward hard constraints

The solver works with double precisions, so it is quite robust to high weight. 10000 is likely to be accepted for example. But if you make the problem too difficult, the solver will break. 
In that case, you can implement a simple penalty solver by setting the weight to be 10**i, and creating a for loop to explore i from 0 to 5. At each iteration of the loop, run the solver from the previous solution and for few iterations only.

In [11]:
for i in range(1,6):
    for m in terminalModels:
        m.differential.costs.costs['gripperPose'].weight = 10**i
    ddp.solve(ddp.xs, ddp.us, 10)

This not very convenient, and a better solver should be used if you really want imposing hard constraints.